In [16]:
import json

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

from torch.utils.data import DataLoader, random_split

In [17]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [18]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset_full = torchvision.datasets.EMNIST(
    root=".",
    split="balanced",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.EMNIST(
    root=".",
    split="balanced",
    train=False,
    download=True,
    transform=transform
)

DATASET_NAME = "EMNIST"
print("train", len(train_dataset_full))
print("test", len(test_dataset))

train 112800
test 18800


In [19]:
SUBSET_FRACTION = 0.5

generator = torch.Generator().manual_seed(SEED)

subset_size = int(len(train_dataset_full) * SUBSET_FRACTION)

train_dataset_full, _ = random_split(
    train_dataset_full,
    [subset_size, len(train_dataset_full) - subset_size],
    generator=generator
)

print("Reduced train full:", len(train_dataset_full))

train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=generator
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

Reduced train full: 56400
Train: 45120
Val: 11280


In [20]:
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [21]:
x_batch, y_batch = next(iter(train_loader))

print("x.shape:", x_batch.shape)
print("y.shape:", y_batch.shape)
print("Min/Max:", x_batch.min().item(), x_batch.max().item())
print("Batch size", BATCH_SIZE)

x.shape: torch.Size([128, 1, 28, 28])
y.shape: torch.Size([128])
Min/Max: 0.0 1.0
Batch size 128


In [22]:
class MLP(nn.Module):
    def __init__(self, input_dim=28*28, hidden_dims=[256], 
                 dropout=0.0, batchnorm=False):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
                
            layers.append(nn.ReLU())
            
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            
            prev_dim = h
        
        layers.append(nn.Linear(prev_dim, 47))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        x = torch.flatten(x, 1)
        return self.model(x)


In [23]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    
    total_loss = 0
    correct = 0
    total = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            
            logits = model(x)
            loss = criterion(logits, y)
            
            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    
    return total_loss / total, correct / total

In [24]:
def run_experiment(
    config,
    epochs=15,
    optimizer_name="Adam",
    lr=1e-3,
    momentum=0.0,
    weight_decay=0.0,
    early_stopping=False,
    patience=3
):
    model = MLP(
        hidden_dims=config["hidden_dims"],
        dropout=config.get("dropout", 0.0),
        batchnorm=config.get("batchnorm", False)
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
    elif optimizer_name == "SGD":
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay
        )
    else:
        raise ValueError("Unknown optimizer")

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    best_val_acc = 0
    best_epoch = 0
    best_weights = None
    patience_counter = 0

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_weights = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if early_stopping and patience_counter >= patience:
            break

    return {
        "config": config,
        "history": history,
        "best_val_acc": best_val_acc,
        "best_val_loss": min(history["val_loss"]),
        "best_epoch": best_epoch,
        "best_weights": best_weights,
        "epochs_trained": len(history["val_acc"]),
        "optimizer": optimizer_name,
        "lr": lr,
        "momentum": momentum,
        "weight_decay": weight_decay
    }

In [25]:
E1 = {"hidden_dims": [512, 256]}
E2 = {"hidden_dims": [512, 256], "dropout": 0.3}
E3 = {"hidden_dims": [512, 256], "batchnorm": True}

results = []

for config in [E1, E2, E3]:
    res = run_experiment(config, epochs=15)
    results.append(res)
    print("Config:", config, "Best val acc:", res["best_val_acc"])

Epoch [1/15] | Train Loss: 1.4029 | Train Acc: 0.6055 | Val Loss: 0.8742 | Val Acc: 0.7382
Epoch [2/15] | Train Loss: 0.7419 | Train Acc: 0.7677 | Val Loss: 0.6957 | Val Acc: 0.7786
Epoch [3/15] | Train Loss: 0.5853 | Train Acc: 0.8096 | Val Loss: 0.6159 | Val Acc: 0.7995
Epoch [4/15] | Train Loss: 0.5018 | Train Acc: 0.8291 | Val Loss: 0.5897 | Val Acc: 0.8017
Epoch [5/15] | Train Loss: 0.4377 | Train Acc: 0.8502 | Val Loss: 0.5428 | Val Acc: 0.8224
Epoch [6/15] | Train Loss: 0.3916 | Train Acc: 0.8602 | Val Loss: 0.5457 | Val Acc: 0.8232
Epoch [7/15] | Train Loss: 0.3519 | Train Acc: 0.8722 | Val Loss: 0.5408 | Val Acc: 0.8229
Epoch [8/15] | Train Loss: 0.3185 | Train Acc: 0.8834 | Val Loss: 0.5499 | Val Acc: 0.8277
Epoch [9/15] | Train Loss: 0.2871 | Train Acc: 0.8906 | Val Loss: 0.5580 | Val Acc: 0.8248
Epoch [10/15] | Train Loss: 0.2632 | Train Acc: 0.8989 | Val Loss: 0.5497 | Val Acc: 0.8293
Epoch [11/15] | Train Loss: 0.2368 | Train Acc: 0.9086 | Val Loss: 0.5959 | Val Acc: 0.82

In [26]:
best_exp_2_3 = max(results[1:], key=lambda x: x["best_val_acc"])
print("Best model config", best_exp_2_3["config"])

e4_res = run_experiment(
    best_exp_2_3["config"],
    epochs=30,
    early_stopping=True,
    patience=5
)
print("Config:", e4_res["config"], "Best val acc:", e4_res["best_val_acc"])
results.append(e4_res)

best_exp = e4_res
print("Best config:", best_exp["config"])
print("Best val_acc:", best_exp["best_val_acc"])

Best model config {'hidden_dims': [512, 256], 'dropout': 0.3}
Epoch [1/30] | Train Loss: 1.6320 | Train Acc: 0.5376 | Val Loss: 0.9140 | Val Acc: 0.7208
Epoch [2/30] | Train Loss: 0.9224 | Train Acc: 0.7160 | Val Loss: 0.7178 | Val Acc: 0.7739
Epoch [3/30] | Train Loss: 0.7614 | Train Acc: 0.7560 | Val Loss: 0.6313 | Val Acc: 0.7949
Epoch [4/30] | Train Loss: 0.6730 | Train Acc: 0.7811 | Val Loss: 0.5978 | Val Acc: 0.8035
Epoch [5/30] | Train Loss: 0.6191 | Train Acc: 0.7947 | Val Loss: 0.5656 | Val Acc: 0.8121
Epoch [6/30] | Train Loss: 0.5710 | Train Acc: 0.8074 | Val Loss: 0.5421 | Val Acc: 0.8198
Epoch [7/30] | Train Loss: 0.5392 | Train Acc: 0.8152 | Val Loss: 0.5207 | Val Acc: 0.8299
Epoch [8/30] | Train Loss: 0.5073 | Train Acc: 0.8248 | Val Loss: 0.5241 | Val Acc: 0.8284
Epoch [9/30] | Train Loss: 0.4863 | Train Acc: 0.8313 | Val Loss: 0.5086 | Val Acc: 0.8346
Epoch [10/30] | Train Loss: 0.4734 | Train Acc: 0.8316 | Val Loss: 0.5201 | Val Acc: 0.8267
Epoch [11/30] | Train Loss:

In [29]:
base_config = e4_res["config"]

res_O1 = run_experiment(
    base_config,
    epochs=7,
    optimizer_name="Adam",
    lr=1e-1
)
res_O2 = run_experiment(
    base_config,
    epochs=7,
    optimizer_name="Adam",
    lr=1e-5
)
res_O3 = run_experiment(
    base_config,
    epochs=12,
    optimizer_name="SGD",
    lr=1e-2,
    momentum=0.9,
    weight_decay=1e-4
)

Epoch [1/7] | Train Loss: 6.0188 | Train Acc: 0.0207 | Val Loss: 3.8693 | Val Acc: 0.0193
Epoch [2/7] | Train Loss: 3.8752 | Train Acc: 0.0206 | Val Loss: 3.8622 | Val Acc: 0.0210
Epoch [3/7] | Train Loss: 3.8691 | Train Acc: 0.0211 | Val Loss: 3.8615 | Val Acc: 0.0225
Epoch [4/7] | Train Loss: 3.8644 | Train Acc: 0.0207 | Val Loss: 3.8606 | Val Acc: 0.0203
Epoch [5/7] | Train Loss: 3.8640 | Train Acc: 0.0218 | Val Loss: 3.8628 | Val Acc: 0.0193
Epoch [6/7] | Train Loss: 3.8649 | Train Acc: 0.0217 | Val Loss: 3.8640 | Val Acc: 0.0209
Epoch [7/7] | Train Loss: 3.8643 | Train Acc: 0.0201 | Val Loss: 3.8622 | Val Acc: 0.0215
Epoch [1/7] | Train Loss: 3.8239 | Train Acc: 0.0483 | Val Loss: 3.7821 | Val Acc: 0.1259
Epoch [2/7] | Train Loss: 3.7176 | Train Acc: 0.1292 | Val Loss: 3.6091 | Val Acc: 0.2630
Epoch [3/7] | Train Loss: 3.4971 | Train Acc: 0.2123 | Val Loss: 3.3172 | Val Acc: 0.3551
Epoch [4/7] | Train Loss: 3.2045 | Train Acc: 0.2689 | Val Loss: 2.9901 | Val Acc: 0.4096
Epoch [5/7

In [30]:
model = MLP(
    hidden_dims=best_exp["config"]["hidden_dims"],
    dropout=best_exp["config"].get("dropout", 0.0),
    batchnorm=best_exp["config"].get("batchnorm", False)
).to(device)

model.load_state_dict(best_exp["best_weights"])

test_loss, test_acc = evaluate(model, test_loader, nn.CrossEntropyLoss())
print("Final TEST accuracy:", test_acc)

Final TEST accuracy: 0.8354787234042553


In [31]:
all_results = {
    "E1": results[0],
    "E2": results[1],
    "E3": results[2],
    "E4": results[3],
    "O1": res_O1,
    "O2": res_O2,
    "O3": res_O3,
}

rows = []

for exp_id, res in all_results.items():
    row = {
        "experiment_id": exp_id,
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": f"hidden={res['config']['hidden_dims']}, "
                         f"dropout={res['config'].get('dropout',0)}, "
                         f"batchnorm={res['config'].get('batchnorm',False)}",
        "optimizer": res["optimizer"],
        "lr": res["lr"],
        "momentum": res["momentum"],
        "weight_decay": res["weight_decay"],
        "epochs_trained": res["epochs_trained"],
        "best_val_accuracy": res["best_val_acc"],
        "best_val_loss": res["best_val_loss"]
    }
    rows.append(row)

df_runs = pd.DataFrame(rows)
df_runs.to_csv("artifacts/runs.csv", index=False)

In [32]:
torch.save(best_exp["best_weights"], "artifacts/best_model.pt")

In [33]:
plt.figure(figsize=(10, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(best_exp["history"]["train_loss"], label="train_loss")
plt.plot(best_exp["history"]["val_loss"], label="val_loss")
plt.title("Loss")
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(best_exp["history"]["train_acc"], label="train_acc")
plt.plot(best_exp["history"]["val_acc"], label="val_acc")
plt.title("Accuracy")
plt.legend()

plt.tight_layout()

plt.savefig("artifacts/figures/curves_best.png")
plt.close()

print("Saved")

Saved


In [36]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(res_O1["history"]["train_loss"], label="O1 train")
plt.plot(res_O1["history"]["val_loss"], label="O1 val")
plt.title("LR too large")

plt.subplot(1,2,2)
plt.plot(res_O2["history"]["train_loss"], label="O2 train")
plt.plot(res_O2["history"]["val_loss"], label="O2 val")
plt.title("LR too small")

plt.legend()
plt.tight_layout()
plt.savefig("artifacts/figures/curves_lr_extremes.png")
plt.close()

print("Saved")

Saved


In [35]:
best_config = {
    "dataset": DATASET_NAME,
    "seed": SEED,
    "hidden_dims": best_exp["config"]["hidden_dims"],
    "dropout": best_exp["config"].get("dropout", 0.0),
    "batchnorm": best_exp["config"].get("batchnorm", False),
    "optimizer": best_exp["optimizer"],
    "learning_rate": best_exp["lr"],
    "momentum": best_exp["momentum"],
    "weight_decay": best_exp["weight_decay"],
    "loss": "CrossEntropyLoss",
    "early_stopping": True,
    "patience": 5
}

with open("artifacts/best_config.json", "w") as f:
    json.dump(best_config, f, indent=4)